# Guardian AI: smart environmental surveillance 

## Scenes classifier

This notebook guides you through building a scene classification system for environmental monitoring using the Places365 dataset. You will begin by selecting relevant environmental scene categories, filtering and preparing the dataset, and then training a CNN from scratch following the provided guidelines. Next, you will apply transfer learning using a pre-trained ResNet18 architecture. The workflow also includes comprehensive model evaluation using confusion matrices and a practical inference system. 

You can use all the materials we have seen during the lectures as reference.
Throughout the notebook, you will find ***questions*** to answer and discuss when presenting your team project results.

### 1. Imports

In [ ]:
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import transforms, datasets

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm  # for progress bar

import PIL.Image as Image

In [ ]:
!which python

In [ ]:
# TODO: For reproducibility, set a seed
SEED = ___

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# TODO:
# Select GPU if available, otherwise use CPU
device = ____

print(f"Using device: {device}")

### 2. Data Preparation

We use the **Places365 dataset**, which contains over **1.8 million images** across **365 environmental scene categories** including forests, deserts, urban areas, and agricultural zones.

Here we define the image preprocessing pipeline with **appropriate transforms** for training and validation data, including data augmentation techniques to make our model robust to real-world conditions.

A list of all the supported computer vision transformations is available in the official documentation here: https://docs.pytorch.org/vision/main/transforms.html


In [ ]:
# Step 1: Set the path to the Places365 dataset

# The path where the Places365 dataset is stored.
places_data_root = "/leonardo_scratch/fast/tra26_bbs/data/places365/"


# Step 2: Define transforms for training data

# Guardian AI must work in different lighting conditions,
# viewpoints, and environmental situations.
#
# We therefore use:
# - data augmentation
# - normalization
#
# TODO: Complete the missing transforms.

train_transforms = transforms.Compose([

    # Randomly crop and resize images to 224x224
    transforms.___(224),

    # Random horizontal flip
    transforms.___(),

    # Convert PIL image to tensor
    transforms.___(),

    # Normalise using ImageNet statistics
    # Mean: [0.485, 0.456, 0.406], Std: [0.229, 0.224, 0.225]
    transforms.___(
        mean=[___, ___, ___],
        std=[___, ___, ___]
    )
])


# Step 3: Define transforms for validation data

# Hint: Validation data should not use random augmentation.
# We only apply deterministic preprocessing.
#
# TODO: Complete the missing transforms.

val_transforms = transforms.Compose([

    # Resize shorter side
    transforms.___(256),

    # Crop centre region
    transforms.___(224),

    # Convert to tensor
    transforms.___(),

    # Normalize images
    transforms.___(
        mean=[___],
        std=[___]
    )
])

The documentation to correctly load the Places365 dataset is available here: https://pytorch.org/vision/stable/generated/torchvision.datasets.Places365.html

In [ ]:
# Step 4: Load Places365 datasets

# TODO: Complete the dataset loading code.

full_train = datasets.___(
    
    # Your dataset path
    root=___,

    # Use "train-standard" for training
    split="___",  
    
    # False for high-resolution images
    small="___", 

    # Apply training transforms
    transform=___,

    download=False
)

full_val = datasets.___(

    # Your dataset path
    root=___,

    # Use "val" for training
    split="___",

    # False for high-resolution images
    small="___",

    # Apply validation transforms
    transform=___,

    download=False
)

### 2.1 Dataset analysis

In [ ]:
# TODO: Verify your datasets
print(f"Training samples: {len(____)}")         # Print the size of the training dataset
print(f"Validation samples: {len(____)}")       # Print the size of the validation dataset
print(f"Number of scene classes: {len(____)}")  # Print the number of the scene classes, hint: use full_train.classes

# Expected output:
# Training samples: ~1,800,000
# Validation samples: ~36,500  
# Number of scene classes: 365

In [ ]:
# Explore Categories
categories = full_train.classes

print("First 20 categories:\n")
for i, cat in enumerate(____):  # TODO: enumerate first 20 categories
    print(f"{i:3d} -> {cat}")

In [ ]:
# Visualise a few random samples
fig, axes = plt.subplots(3, 3, figsize=(12,12))

# TODO: Randomly select 9 image indices from the dataset
random_indices = ____

for ax, idx in zip(axes.flatten(), random_indices):
    
    # TODO: Retrieve image and label
    img, label = ____

    # Denormalise image for visualisation

    # Change tensor format from (C,H,W) to (H,W,C)
    img = img.permute(1,2,0).numpy()

    
    # ImageNet normalisation statistics
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    img = std * img + mean

    # Clamp values between 0 and 1
    img = np.clip(img, 0, 1)

    ax.imshow(img)

     # Extract category name
    category = categories[label].split("/")[-1]

    ax.set_title(category)
    ax.axis("off")

plt.suptitle("Random Samples from Places365")
plt.tight_layout()
plt.show()

In [ ]:
# QUESTION:
# Are all the images correctly described by their corresponding labels? What observations can you make, and why do you think this is?

In [ ]:
# Visualise the number of images per category using a random subset
# Since computing the exact class distribution over 1.8 million images can be slow.

sample_size = 20000 

# TODO: Randomly select 20000 image indices from the dataset
sample_indices = ___

sample_labels = []

for idx in tqdm(sample_indices):

    _, label = full_train[idx]

    sample_labels.append(label)

label_counts = Counter(sample_labels)

most_common = label_counts.most_common(15)

top_classes = [categories[idx].split("/")[-1] for idx, _ in most_common]
top_counts = [count for _, count in most_common]

plt.figure(figsize=(12,5))

plt.bar(___, ____) # TODO: Show the bar plot of the top classes using the estimates in top_counts

plt.xticks(rotation=45, ha="right")

plt.ylabel("Frequency in Sample")
plt.title("Most Common Categories in Places365 Sample")

plt.show()

In [ ]:
# Check image resolutions
widths = []
heights = []

# TODO: Randomly select 1000 image indices from the dataset
sample_indices = ___

for idx in tqdm(sample_indices):

    try:
        path, _ = full_train.imgs[idx]

        img = Image.open(path)

        w, h = img.size

        widths.append(w)
        heights.append(h)

    except:
        pass

print(f"Average width:  {___}")  # TODO: Print the average width
print(f"Average height: {___}")  # TODO: Print the average height

print(f"Min width: {___}")       # TODO: Print the min width
print(f"Max width: {___}")       # TODO: Print the max width

print(f"Min height: {___}")       # TODO: Print the min height
print(f"Max height: {___}")       # TODO: Print the max height

In [ ]:
# TODO: Plot Resolution Distribution
plt.figure(figsize=(8,6))
# Add plotting of width and height
plt.scatter(___, ___, alpha=0.3)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Resolution Distribution")
plt.show()

In [ ]:
# QUESTION: 
# Do all the images have the same resolution? Do you think this could cause an issue?

### 3. Categories selection

For our Guardian AI monitoring system, we don't need all the 365 scene categories - instead, we focus on the most relevant environmental contexts where protected area monitoring typically occurs. In this step, we **select approximately 15-18 scene categories** that represent the diverse ecosystems and terrains our drones and camera traps will encounter.

In [ ]:
# Step 1: Explore available categories in Places365
categories = full_train.classes
print("Sample categories from Places365:")
print(categories[:10])
print(f"Total available categories: {len(categories)}")

In [ ]:
# Hint: You can search for specific environments like this:
#
# forest_categories = [cat for cat in categories if 'forest' in cat]
# print("Forest categories:", forest_categories)
#
# or find all water-related environments:
#
# water_cats = [cat for cat in categories if any(word in cat.lower() for word in ['lake', 'river', 'ocean', 'coast', 'beach'])]
# print(water_cats)

In [ ]:
# Step 2: Select 15-18 environmental categories relevant for protected area monitoring
# Think about different ecosystems where environmental monitoring is crucial

selected_categories = [
    # TODO: Add your chosen categories here
    # Example format: '/f/forest/broadleaf'
    
    '___', 
    '___',
    '___',
    '___',
    '___',
    '___',

]

# TODO: Step 3: Define the number of classes you selected and verify all categories exist in the dataset
num_classes = ___
selected_category_indices = ___

In [ ]:
# Step 4: Verify that all selected categories exist in the dataset
selected_category_indices = [i for i, cat in enumerate(categories) if cat in selected_categories]
print(f"Found {len(selected_category_indices)} valid categories in dataset")

In [ ]:
# Step 5: Display selected categories with their indices
print("\nSelected Environmental Categories:")
for i, cat in enumerate(selected_categories):
    if cat in categories:
        idx = categories.index(cat)
        print(f"{i+1:2d}. {cat} (dataset index: {idx})")
    else:
        print(f"{i+1:2d}. {cat} - NOT FOUND!")

### 4. Filtering dataset indices for selected categories

Now that we've chosen our environmental categories, we need to identify which images in the massive Places365 dataset belong to our selected categories. With over 1.8 million training images, checking each one sequentially would take hours. Instead, we'll use parallel processing with ThreadPoolExecutor to speed up this filtering process. We create a function that checks if each image belongs to one of our selected categories, then run it across multiple CPU cores simultaneously. The result is a list of indices pointing to only the images we need for training our Guardian AI system, which we'll save to disk to avoid repeating this time-intensive process.

- Have a look at the notebook `03-python_speedup.ipynb` to recap how multithreading works.

In [ ]:
def is_selected_train(i):
    """
    Function to check if training image at index i belongs to our selected categories.
    
    Args:
        i (int): Index of the image in full_train dataset
        
    Returns:
        int or None: Returns i if image belongs to selected categories, None otherwise
    """
    try:
        # TODO: Get the image and label from full_train at index i
        path, label = ___ # Hint: Use full_train.imgs[i] and unpack with path, label = ...
        
        # TODO: Check if this label is in our selected_category_indices and return i if yes, None if no
        if ___:
            return ___
        else:
            ___
        
    except Exception:
        # Handle corrupted images or other errors
        return None

In [ ]:
# Use ThreadPoolExecutor to process training set in parallel
with ThreadPoolExecutor(max_workers=___) as executor: # TODO: Set the number of threads
    
    # TODO: Map is_selected_train function across all training indices, how can you obtain the range of indices to test?
    # Use tqdm for progress bar
    selected_indices_train = list(tqdm(
        executor.map(___, ___), 
        total=len(full_train),
        desc="Training"
    ))

# TODO: Remove None values from the results
# Hint: Use list comprehension to keep only non-None values
selected_indices_train = [___ for ___ in ___ if ___ is not None]

print(f"Found {len(selected_indices_train):,} training images from selected categories")

In [ ]:
def is_selected_val(i):
    """
    Function to check if validation image at index i belongs to our selected categories.
    Args:
        i (int): Index of the image in full_val dataset
        
    Returns:
        int or None: Returns i if image belongs to selected categories, None otherwise
    """
    try:
        # TODO: Similar to above but use full_val dataset
        path, label = ___
        
    except Exception:
        return None

In [ ]:
with ThreadPoolExecutor(max_workers=___) as executor:  # TODO: Set the number of threads (use fewer workers for smaller dataset)
    selected_indices_val = list(tqdm(
        executor.map(___, ___),  # TODO: Map is_selected_val function across all validation indices
        total=len(full_val),
        desc="Validation"
    ))

# TODO: Filter out None values
selected_indices_val = [___ for ___ in ___ if ___ is not None]

print(f"Found {len(selected_indices_val):,} validation images from selected categories")

In [ ]:
# TODO: Save the indices to avoid recomputing this expensive operation
np.save("___", selected_indices_train)  # Save it in your own folder
np.save("___", selected_indices_val)    # Save it in your own folder

In [ ]:
# For future notebook runs, you can load saved indices instead:
# selected_indices_train = np.load("<path_to_your_own_folder>/selected_indices_train.npy").tolist()
# selected_indices_val = np.load("<path_to_your_own_folder>/selected_indices_val.npy").tolist()

If the training set is too large and you want to reduce its size in order to test the entire code before the final training run, you can randomly select half of the indices or a portion of them.

In [ ]:
reduced_size = ____ # TODO:  Choose the reduced size
reduced_indices = random.sample(_____, ______) # TODO: Select the random indices from the training indices set

### 5. Filter dataset and label remapping

After identifying relevant image indices, we need to create filtered datasets containing only our selected environmental categories. However, there's a crucial issue: the original Places365 labels range from 0-364, but our 18 selected categories will have scattered indices (e.g., 45, 123, 200, etc.). Machine learning models expect consecutive labels starting from 0, so we must remap our categories from their original scattered indices to a clean 0-17 range. We also create a custom Dataset class that handles this remapping automatically. 

In [ ]:
# Step 1: filter the dataset
# TODO: Create subsets using the filtered indices
filtered_subset_train = Subset(___, ___)  # Use full_train with the selected indices (or reduced_indices)
filtered_subset_val = Subset(___, ___)    # Use full_val with selected_indices_val 

In [ ]:
# Step2: implement custom Dataset class for label remapping
class RemappedSubset(Dataset):
    """
    Why do we need remapping?
    - Original Places365 has labels like [5, 23, 156, 299] for our selected categories
    - ML models need consecutive labels [0, 1, 2, 3] for proper training
    - This class automatically converts scattered labels to consecutive ones
    """
    
    def __init__(self, subset, selected_categories):
        # TODO: Store the indices and original dataset
        self.indices = subset.indices.copy()  # Copy the list of indices
        self.orig_dataset = ___               # Store reference to original dataset
        
        # TODO: Create mapping dictionary from category path to new index
        # category_name -> new_label
        # Hint: {category: index for index, category in enumerate(selected_categories)}
        self.cat2idx = {___: ___ for ___, ___ in enumerate(___)}
        self.categories = selected_categories
        
        print(f"Created remapping for {len(selected_categories)} categories")
    
    def __len__(self):
        # TODO: Return the length of indices
        return len(___)
    
    def __getitem__(self, idx):        
        # TODO: Get the original index and retrieve image + label
        original_index = self.indices[___]              # Get true index
        img, label = self.orig_dataset[___]             # Get image and original label
        
        # TODO: Convert original label to category path, then to new label
        class_path = self.orig_dataset.classes[___]     # Get category string
        new_label = self.cat2idx[___]                   # Map to new consecutive label
        
        return img, new_label

In [ ]:
# Step 3: Create the remapped datasets
train_dataset = RemappedSubset(___, ___)  # TODO: Use the filtered dataset for training and the list of the selected categories
val_dataset = RemappedSubset(___, ___)    # TODO: Use the filtered dataset for validation and the list of the selected categories

In [ ]:
# Step 4: Show the sizes of the obtained datasets
print(f"Final dataset sizes:")
print(f"Training: {len(___)} images")                   # TODO: Print the size of the training dataset
print(f"Validation: {len(___)} images")                 # TODO: Print the size of the validation dataset
print(f"Classes: {len(___)} environmental categories")  # TODO: Print the number of the selected categories

In [ ]:
# Step 5: Verify that label remapping works correctly
unique_labels = set()
sample_size = min(500, len(train_dataset))

# TODO: Sample some images and collect their new labels
for i in random.sample(range(len(train_dataset)), k=sample_size):
    _, label = ___[___]  # Get label from train_dataset
    unique_labels.add(___)

print(f"Found labels: {sorted(unique_labels)}")
print(f"Expected: {list(range(len(selected_categories)))}")

### 5.1 Dataset analysis

In [ ]:
# TODO: Count labels in train dataset

# Extract labels from the training and validation datasets
train_labels = ___
val_labels = ___

train_counts = Counter(train_labels)
val_counts = Counter(val_labels)

distribution_df = pd.DataFrame({
    "Category": selected_categories,
    "Train Samples": [train_counts[i] for i in range(num_classes)],
    "Validation Samples": [val_counts[i] for i in range(num_classes)]
})

distribution_df

In [ ]:
# TODO: Plot class distribution
fig, ax = plt.subplots(figsize=(14,6))

x = np.arange(num_classes)

ax.bar(x - 0.2,
       ___,  # Plot the number of the training samples from distribution_df
       width=0.4,
       label="Train")

ax.bar(x + 0.2,
       ___,  # Plot the number of the validation samples from distribution_df
       width=0.4,
       label="Validation")

ax.set_xticks(x)
ax.set_xticklabels(
    [c.split("/")[-1] for c in selected_categories],
    rotation=45,
    ha="right"
)

ax.set_ylabel("Number of Images")
ax.set_title("Dataset Class Distribution")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualise samples per environmental category

# ImageNet normalization statistics
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

# Step 1: Define a denormalisation function

# TODO: Reverse image normalisation so images can be displayed correctly.
def denormalize(img_tensor):
    return ___

# Step 2: Build a dictionary:

# class_label -> list of dataset indices
class_to_indices = defaultdict(list)

for i in range(len(train_dataset)):
    # TODO: Retrieve image and label from dataset
    _, label = ___

    # TODO: Store the sample index in the dictionary
    class_to_indices[___].append(i)

# Step 3: Visualize random samples for each category

samples_per_class = 3

fig, axes = plt.subplots(
    len(class_to_indices),
    samples_per_class,
    figsize=(12, len(class_to_indices) * 2)
)

axes = np.atleast_2d(axes)

for row, (class_idx, indices) in enumerate(class_to_indices.items()):

    # TODO: Randomly select images from this class
    chosen = ____

    for j, idx in enumerate(chosen):

        img, label = train_dataset[idx]

        img = (
            denormalize(img)  # Reverse normalization
            .permute(1,2,0)   # Convert tensor shape:
            .clamp(0,1)       # Clamp values between 0 and 1
            .numpy()          # Convert to NumPy
        )

        axes[row, j].imshow(img)
        axes[row, j].axis("off")

        # Display category name only on middle image
        if j == 1:
            axes[row, j].set_title(
                selected_categories[label].split("/")[-1]
            )

plt.tight_layout()
plt.show()

In [ ]:
# Visualise data augmentation

# TODO: Randomly select one image from the training dataset
idx = ___
img, label = ___

# Reverse normalisation for visualisation

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

def denormalize(x):
    x = x.permute(1, 2, 0).numpy()
    return np.clip(std * x + mean, 0, 1)

img_vis = denormalize(img)

# Convert NumPy image -> PIL image
# Data augmentation pipelines in torchvision
# usually expect PIL images as input.
img_pil = Image.fromarray((img_vis * 255).astype(np.uint8))

# Apply augmentation multiple times
n_aug = 6

fig, axes = plt.subplots(1, n_aug, figsize=(15, 4))

for i in range(n_aug):
    
    # TODO: Apply training augmentation pipeline
    aug_img = ___

    # denormalise for visualisation
    aug_img = aug_img.permute(1, 2, 0).numpy()
    # Reverse normalisation
    aug_img = std * aug_img + mean
    aug_img = np.clip(aug_img, 0, 1)

    axes[i].imshow(aug_img)
    axes[i].axis("off")

    if i == 0:
        axes[i].set_title(
            categories[label].split("/")[-1]
        )

plt.suptitle("Data Augmentation Examples")
plt.tight_layout()
plt.show()

In [ ]:
# QUESTION:
# Why use data augmentation?

In [ ]:
# Estimate the imbalance ratio of the training dataset

# TODO: Extract sizes of the classes from the training dataset
train_class_sizes = ___

imbalance_ratio = train_class_sizes.max() / train_class_sizes.min()

print(f"Imbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print("Dataset is imbalanced")
else:
    print("Dataset is reasonably balanced")

In [ ]:
# QUESTION:
# Is your dataset balanced or imbalanced? Why does this matter?

### 6. Model definition

In the following part, we define our vision model using PyTorch. The SimpleCNN class implements a convolutional neural network with:

- Three convolutional layers (32, 64, 128 filters) with ReLU activation and max pooling for feature extraction
- Two fully connected layers with dropout for classification
- A forward pass that transforms input images into class scores for scene prediction
  
The model takes 224x224 RGB images and outputs class probabilities for the environment classification selected above, such as forest, urban, desert, and agricultural zones.

Have a look at the notebook `01_TrainingFromScratch.ipynb` to recap how to define a model and how to train it.

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
class SimpleCNN(nn.Module):
    """
    Simple CNN for scene classification.
    """

    def __init__(self, num_classes):

        super().__init__()

        # Convolutional feature extractor
        
        self.features = nn.Sequential(

            # Block 1
            # TODO: Add a convolutional layer with ___ input channels -> 32 output channels, 3x3 kernel, padding=1
            nn.___(
                in_channels=___, # How many channels do the input images have?
                out_channels=___,
                kernel_size=___,
                padding=___
            ),
            nn.BatchNorm2d(32),
            # TODO: Add ReLU activation
            nn.___,
            # TODO: Add MaxPool2d layer with kernel size 2
            nn.___,

            # Block 2
            # TODO: Add a convolutional layer with ___ input channels -> 64 output channels, 3x3 kernel, padding=1
            nn.___(
                in_channels=___, # How many channels do the images have after the first convolutional layer?
                out_channels=___,
                kernel_size=___,
                padding=___
            ),
            nn.BatchNorm2d(64),
            # TODO: Add ReLU activation
            nn.___,
            # TODO: Add MaxPool2d layer with kernel size 2
            nn.___,

            # Block 3
            # TODO: Add a convolutional layer with ___ input channels -> 128 output channels, 3x3 kernel, padding=1
            nn.___(
                in_channels=___, # How many channels do the images have after the second convolutional layer?
                out_channels=___,
                kernel_size=___,
                padding=___
            ),
            nn.BatchNorm2d(128),
            # TODO: Add ReLU activation
            nn.___,
            # TODO: Add MaxPool2d layer with kernel size 2
            nn.___,

            # Block 4
            # TODO: Add a convolutional layer with ___ input channels -> 256 output channels, 3x3 kernel, padding=1
            nn.___(
                in_channels=___, # How many channels do the images have after the third convolutional layer?
                out_channels=___,
                kernel_size=___,
                padding=___
            ),
            nn.BatchNorm2d(256),
            # TODO: Add ReLU activation
            nn.___,

            # TODO: Global Average Pooling 
            # Use AdaptiveAvgPool2d with output_size=(1, 1)
            ____
        )

        # Classifier
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # TODO: Add a fully connected layer with 256 ->  128 neurons
            ___
            # TODO: Add ReLU activation
            nn.___,
            nn.Dropout(0.5),
            # TODO: Add a fully connected layer with 128 -> num_classes neurons
            ___
        )

    def forward(self, x):
        # TODO: Define the forward pass: input image -> features -> classification
        ___
        ___

        return ___

In [ ]:
# QUESTION: 
# Why are CNNs useful for computer vision tasks?

### 7. Training loop

Now we are ready to train our scene classification model using a standard PyTorch training loop. The training pipeline includes:

- **DataLoaders**: efficient mini-batch loading for training and validation data
- **Loss function**: CrossEntropyLoss for multi-class classification
- **Optimiser**: Adam optimiser for gradient-based learning
- **Training loop**: forward pass, loss computation, backpropagation, and parameter updates
- **Validation loop**: evaluation on unseen validation data after each epoch
- **Checkpoint saving**: automatic saving of the model with the best validation accuracy

During training, the model learns to recognise different environmental scenes by minimising classification error on the training set while monitoring generalisation performance on the validation set.

The training process runs on a GPU when available to accelerate computation.

In [ ]:
# Training Guardian AI Scene Classifier

# Step 1: Create DataLoaders

train_loader = ___(
    ___, # TODO: Insert training dataset

    batch_size=___, # TODO: Choose an appropriate value for the batch_size 

    shuffle=___, # Shuffle training data

    num_workers=8,

    pin_memory=True,

)

val_loader = ___(
    ___, # TODO: Insert validation dataset

    batch_size=___, # TODO: Choose an appropriate value for the batch_size 

    shuffle=___, # Validation data should not be shuffled

    num_workers=8,

    pin_memory=True
)

# Step 2: Initialise model

# TODO: Pass the number of classes from selected_categories
model = ___

# TODO: Move model to device
model = ___

# Step 3: Define loss function and optimiser

criterion = ___ # TODO: Define the loss function (choose a loss function appropricate for a multi-class classification task)

optimizer = ___ # TODO: Define the optimizer (e.g., Adam optimizer)

# Step 4: Training configuration

num_epochs = 30

best_val_acc = 0.0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Top-5 accuracy checks whether the correct class
# appears among the model's 5 highest-confidence predictions.
# This metric is commonly used in large-scale image
# classification benchmarks such as ImageNet.
val_top5_accuracies = []

# Step 5: Start training

t0 = time.time()

for epoch in range(num_epochs):

    # Training phase
    
    # TODO: Set model to training mode
    ___

    running_loss = 0.0
    correct = 0
    total = 0

    train_bar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for inputs, labels in train_bar:
        
        # TODO: Move tensors to device
        inputs = ___
        labels = ___

        # Forward and backward pass

        # TODO: Reset gradients
        ____

        # TODO: Forward pass
        outputs = ___

        # TODO: Compute classification loss
        loss = ___

        # TODO: Backpropagation
        ____

        # TODO: Update parameters
        ____

        running_loss += loss.item()

        # TODO: Compute predictions
        preds = ___

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        train_bar.set_postfix({
            "loss": loss.item(),
            "acc": correct / total
        })

    # Epoch training statistics

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validation phase
    
    # TODO: Set model to evaluation mode
    ____

    val_loss = 0.0
    correct = 0
    total = 0
    top5_correct = 0

    # Disable gradient computation
    with ___:

        for inputs, labels in val_loader:

            # TODO: Move inputs and labels to device
            inputs = ___
            labels = ___

            # TODO: Forward pass
            outputs = ___

            # TODO: Validation loss
            loss = ___

            val_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            
            # Top-5 predictions
            top5_preds = torch.topk(outputs, 5, dim=1).indices

            top5_correct += (top5_preds == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    # Validation statistics

    val_loss /= len(val_loader)
    val_acc = correct / total
    val_top5_acc = top5_correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_top5_accuracies.append(val_top5_acc)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"Val Top-5 Acc: {val_top5_acc:.4f}")

    # Save best model
    
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        # TODO: Save model weights
        ____

        print("Best model saved")

t1 = time.time()

print(f"\nTraining completed in {(t1 - t0):.2f} seconds")
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Visualise training history

epochs = range(1, num_epochs + 1)

plt.figure(figsize=(14,5))

# Loss
plt.subplot(1,2,1)
plt.plot(___, ___, label="Train Loss")  # TODO: Plot training loss for each epoch
plt.plot(___, ___, label="Val Loss")    # TODO: Plot validation loss for each epoch
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(___, ___, label="Train Accuracy")       # TODO: Plot training accuracy for each epoch
plt.plot(___, ___, label="Validation Accuracy")  # TODO: Plot validation accuracy for each epoch
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Print training time
print('Training time: ', t1-t0, 's')

In [ ]:
# TODO: Compute the number of parameters in the model
num_params = ___
print(f"{num_params:,} parameters")

In [ ]:
# QUESTION:
# The architecture of the SimpleCNN model is intentionally simple to help illustrate the core components 
# of convolutional neural networks and the training process in computer vision. 
# Do you still think its performance is good enough? Why?
# Why is it important to observe the top-5 accuracy to assess the performance of our classifier?

### 8. Evaluation and confusion matrix

Once training is complete, we evaluate the model and visualise its performance using a confusion matrix. The evaluation process includes:

- **Model loading**: loading the best saved model weights or using the trained model directly
- **Inference mode**: disabling gradient computation for faster evaluation
- **Prediction collection**: generating predictions on the validation dataset
- **Accuracy calculation**: computing overall classification accuracy
- **Confusion matrix visualisation**: analysing correct and incorrect predictions for each environmental category

The confusion matrix helps us understand which scene categories the model easily recognises and which it commonly confuses. This analysis is particularly important for environmental monitoring systems such as Guardian AI, where reliable scene recognition is essential for real-world deployment.

In [ ]:
# Model loading (choose one approach):

# OPTION 1: Load saved model weights
# Uncomment and modify the path if you saved a checkpoint during training

# model = SimpleCNN(num_classes=len(selected_categories)).to(device)
# model.load_state_dict(torch.load("best_model.pth"))
# model.eval()

# If the device is still not defined
# TODO: Set up device for evaluation
# device = ___
# print(f"Using device: {device}")

# OPTION 2: Use the already-trained model from the previous section
# (No additional loading required)

# TODO: Move model to device and set to evaluation mode
___
___

# Gather predictions & labels
all_preds = []
all_labels = []

# TODO: Disable gradient computation for evaluation
with ___:
    for inputs, labels in val_loader:
        # TODO: Move inputs and labels to device
        inputs = ___
        labels = ___
        
        # TODO: Get model predictions
        outputs = ___
        
        # TODO: Get predicted classes (use torch.max to find the class with highest probability)
        _, preds = ___
        
        # TODO: Store predictions and labels (convert to CPU numpy arrays)
        all_preds.extend(___)
        all_labels.extend(___)

# TODO: Convert lists to numpy arrays
all_preds = ___
all_labels = ___

# TODO: Compute overall accuracy
overall_acc = ___
print(f"Overall Validation Accuracy: {overall_acc:.4f}")

# TODO: Create confusion matrix
cm = ___

# TODO: Create confusion matrix display
disp = ___

# TODO: Create and show the plot
fig, ax = plt.subplots(figsize=(12, 12))
# TODO: Plot the confusion matrix
___
plt.title("Confusion Matrix for Scene Classification")
plt.show()

In [ ]:
# TODO: Estimate per-class accuracy = diagonal / row sum
class_acc = ___

per_class_df = pd.DataFrame({
    "Category": [c.split("/")[-1] for c in selected_categories],
    "Accuracy": class_acc
})

per_class_df = per_class_df.sort_values("Accuracy", ascending=False)

per_class_df

plt.figure(figsize=(14, 5))

plt.bar(
    range(len(class_acc)),
    class_acc
)

plt.xticks(
    range(len(class_acc)),
    [c.split("/")[-1] for c in selected_categories],
    rotation=45,
    ha="right"
)

plt.ylabel("Accuracy")
plt.title("Per-Class Validation Accuracy")
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Print top 5 easiest classes
print("\nEasiest classes:")
print(___)

# TODO: Print top 5 hardest classes
print("\nHardest classes:")
print(___)

In [ ]:
# Analyse the most confused scene categories
pairs = []

num_classes = cm.shape[0]

# Loop through all class combinations
for i in range(num_classes):
    for j in range(num_classes):
        if i != j:
            # TODO: Append a tuple containing:
            # - true class index
            # - predicted class index
            # - number of confusions from cm
            pairs.append((
                ___,
                ___,
                ___
            ))

# Sort pairs by confusion count (highest first)

# TODO: Sort the list using the confusion count
pairs = sorted(
    pairs,
    key=lambda x: ___,
    reverse=True
)

# Create DataFrame with top confused pairs

confusion_pairs_df = pd.DataFrame(

    # TODO: Keep only the top 20 most confused pairs
    ___,

    columns=[
        "True Class",
        "Predicted Class",
        "Count"
    ]
)

# Convert class indices into readable labels

confusion_pairs_df["True Label"] = confusion_pairs_df["True Class"].apply(

    # TODO: Extract category name from selected_categories
    lambda x: ______________________________
)

confusion_pairs_df["Predicted Label"] = confusion_pairs_df["Predicted Class"].apply(

    # TODO: Extract category name from selected_categories
    lambda x: ______________________________
)

# Keep only useful columns
confusion_pairs_df = confusion_pairs_df[
    ["True Label", "Predicted Label", "Count"]
]

# Display table
confusion_pairs_df

In [ ]:
# Visualise top confusion pairs

plt.figure(figsize=(12, 6))

# Create labels such as:
# forest → mountain
labels = [
    f"{t} → {p}"
    for t, p in zip(
        confusion_pairs_df["True Label"],
        confusion_pairs_df["Predicted Label"]
    )
]

# Confusion counts
counts = confusion_pairs_df["Count"]

# TODO: Create horizontal bar plot
plt.___(labels[::-1], counts[::-1])

plt.xlabel("Number of Confusions")
plt.title("Top Confused Class Pairs")

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Estimate normalise confusion matrix
cm_norm = ___ # Avoid division by zero

plt.figure(figsize=(10, 8))

sns.heatmap(
    ___,  # TODO: Plot the normalized confusion matrix
    xticklabels=[c.split("/")[-1] for c in selected_categories],
    yticklabels=[c.split("/")[-1] for c in selected_categories],
    cmap="Blues"
)

plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Estimate normalise confusion pairs
pairs_norm = []

for i in range(num_classes):
    for j in range(num_classes):
        if i != j:
            pairs_norm.append((i, j, cm_norm[i, j]))

pairs_norm = sorted(pairs_norm, key=lambda x: x[2], reverse=True)
pairs_norm

In [ ]:
# QUESTION:
# Are some scene classes much harder to classify than others? Why?

### 9. Sample Inference

Here we can check how Guardian AI performs real-world scene classification on individual images. 

In [ ]:
def predict_scene(image_path, model, transform, class_names):
    img = Image.open(image_path).convert("RGB")
    inp = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():
        out = model(inp)
        _, pred = torch.max(out, 1)
    return class_names[pred.item()]

# TODO: Randomly select 5 images from our filtered validation indices
sample_indices = ___
test_images = []
true_labels = []

for idx in sample_indices:
    # TODO: Get the image path and true label from the original validation dataset
    img_path, true_class_idx = ___
    test_images.append(img_path)
    true_labels.append(categories[true_class_idx])

print(f"Testing on 5 random images from our selected validation categories...")
print(f"Selected indices: {sample_indices}")

correct_predictions = 0
for i, (img_path, true_label) in enumerate(zip(test_images, true_labels)):
    try:
        predicted_label = predict_scene(img_path, model, val_transforms, selected_categories)
        img = Image.open(img_path).convert("RGB")
        
        # TODO: Check if prediction is correct
        is_correct = ___
        if is_correct:
            correct_predictions += 1
            
        plt.figure(figsize=(6,6))
        plt.imshow(img)
        title_color = 'green' if is_correct else 'red'
        plt.title(f"Sample {i+1}\nTrue: {true_label}\nPredicted: {predicted_label}", 
                 fontsize=12, fontweight='bold', color=title_color)
        plt.axis("off")
        plt.show()
        
        status = f"\nCORRECT" if is_correct else f"\nINCORRECT"
        print(f"Image: {img_path.split('/')[-1]}")
        print(f"True Label: {true_label}")
        print(f"Predicted: {predicted_label} - {status}")
        print("-" * 50)
    except Exception as e:
        print(f"Error processing image {img_path}: {e}")

print(f"\nSample Accuracy: {correct_predictions}/{len(test_images)} = {correct_predictions/len(test_images):.2%}")

### 10. Transfer Learning and Fine-Tuning

We now use a ResNet-18 model pretrained on the ImageNet dataset.

**Transfer learning** is a machine learning technique in which knowledge learned from one task is reused for another related task. Instead of training a neural network from scratch, we start from a model that has already learned useful visual features from millions of images.

In this notebook, the pretrained ResNet-18 already knows how to detect generic visual patterns such as edges, textures, shapes, and object parts. We adapt the model to our environmental scene classification task by replacing the final classification layer.

During the first training phase, we **freeze** all pretrained layers and train only the new classifier. In this way, the pretrained feature extractor remains unchanged while the final layer learns to map extracted features to our 18 scene categories.

This approach is called **transfer learning**.

Afterwards, we optionally perform **fine-tuning**. In fine-tuning, some pretrained layers are unfrozen and updated with a very small learning rate. This allows the pretrained representations to slightly adapt to the new dataset while preserving the useful knowledge learned from ImageNet.

In summary:

* **Transfer learning:** train only the newly added classifier while keeping pretrained layers frozen.
* **Fine-tuning:** unfreeze part of the pretrained network and continue training with a small learning rate.

Have a look at section 5 of the notebook `extra_comparison_different_techniques.ipynb` to recap the implementation of transfer learning.

In [ ]:
from torchvision.models import resnet18

In [ ]:
model = resnet18()
model.load_state_dict(torch.load("/leonardo_scratch/fast/tra26_bbs/models/resnet/resnet18-f37072fd.pth"))

In [ ]:
# TODO: Move model to device
model = ___

In [ ]:
# TODO: Replace final classifier layer
# with a new classifier for the selected classes
model.fc = ___

In [ ]:
# TODO: Freeze backbone
# This means to freeze all pretrained layers:
# their weights will NOT be updated during backpropagation
for param in model.parameters():
    ___

# TODO: Unfreeze only the final classification layer:
# this is the only part trained during transfer learning
for param in model.fc.parameters():
    ___

In [ ]:
# QUESTION:
# Why do we freeze pretrained layers during transfer learning?

In [ ]:
# Define optimiser only on trainable params with requires_grad=True
# (currently only the final classifier layer)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

#### Optional fine-tuning phase
***Important***: Fine-tuning is usually performed after training the new classifier layer.
First, the model learns how to use pretrained features for the new task.
Then, selected pretrained layers are slightly adjusted to better fit the target dataset.

In [ ]:
# After transfer learning, we unfreeze part of the pretrained
# network to slightly adapt learned features to our dataset
fine_tuning = False

if fine_tuning == True:
    
    # TODO: Unfreeze last ResNet block
    # so its weights can be updated during training
    for param in model.layer4.parameters():
        ___

    # Use different learning rates:
    # - smaller learning rates for pretrained layers (careful updates)
    # - larger learning rates for the new classifier (trained from scratch)
    optimizer = optim.AdamW(
        [
            {
                "params": model.layer4.parameters(),
                "lr": 1e-5
            },
            {
                "params": model.fc.parameters(),
                "lr": 1e-4
            }
        ],
        weight_decay=1e-4
    )
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Learning rate scheduler:
    # reduce learning rate when validation accuracy plateaus
    # to stabilise fine-tuning
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

In [ ]:
# Training loop stays the same, we only add the learning rate scheduler for fine-tuning

# Step 1: Create DataLoaders

train_loader = ___(

    ___, # TODO: Insert training dataset

    batch_size=___, # TODO: Choose an appropriate value for the batch_size 

    shuffle=___, # Shuffle training data

    num_workers=8,

    pin_memory=True,

)

val_loader = ___(

    ___, # TODO: Insert validation dataset

    batch_size=___, # TODO: Choose an appropriate value for the batch_size
    
    shuffle=___, # Validation data should not be shuffled

    num_workers=8,

    pin_memory=True
)

# Step 2: Training configuration

num_epochs = 30

best_val_acc = 0.0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
val_top5_accuracies = []

# Step 3: Start training

t0 = time.time()

for epoch in range(num_epochs):

    # Training phase
    
    # TODO: Set model to training mode
    ___

    running_loss = 0.0
    correct = 0
    total = 0

    train_bar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for inputs, labels in train_bar:
        
        # TODO: Move tensors to device
        inputs = ___
        labels = ___

        # Forward and backward pass

        # TODO: Reset gradients
        ____

        # TODO: Forward pass
        outputs = ___

        # TODO: Compute classification loss
        loss = ___

        # TODO: Backpropagation
        ____

        # TODO: Update parameters
        ____

        running_loss += loss.item()

        # TODO: Compute predictions
        preds = ___

        correct += (preds == labels).sum().item()

        total += labels.size(0)

        train_bar.set_postfix({
            "loss": loss.item(),
            "acc": correct / total
        })

    # Epoch training statistics

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total
    
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Validation phase
    
    # TODO: Set model to evaluation mode
    ____

    val_loss = 0.0
    correct = 0
    total = 0
    top5_correct = 0

    # Disable gradient computation
    with ___:

        for inputs, labels in val_loader:

            # TODO: Move inputs and labels to device
            inputs = ___
            labels = ___

            # TODO: Forward pass
            outputs = ___

            # TODO: Validation loss
            loss = ___

            val_loss += loss.item()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            
            # Top-5 predictions
            top5_preds = torch.topk(outputs, 5, dim=1).indices

            top5_correct += (top5_preds == labels.unsqueeze(1)).any(dim=1).sum().item()

            total += labels.size(0)

    # Validation statistics

    val_loss /= len(val_loader)
    val_acc = correct / total
    val_top5_acc = top5_correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_top5_accuracies.append(val_top5_acc)

    
    # For fine-tuning, we use the earning rate scheduler
    if fine_tuning == True:
        scheduler.step(val_acc)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"Val Top-5 Acc: {val_top5_acc:.4f}")

    # Save best model
    
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        # TODO: Save model weights
        ____

        print("Best model saved")

t1 = time.time()

print(f"\nTraining completed in {(t1 - t0):.2f} seconds")
print(f"Best validation accuracy: {best_val_acc:.4f}")

In [ ]:
# Visualise training history

epochs = range(1, num_epochs + 1)

plt.figure(figsize=(14,5))

# Loss
plt.subplot(1,2,1)
plt.plot(___, ___, label="Train Loss")  # TODO: Plot training loss for each epoch
plt.plot(___, ___, label="Val Loss")    # TODO: Plot validation loss for each epoch
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(___, ___, label="Train Accuracy")       # TODO: Plot training accuracy for each epoch
plt.plot(___, ___, label="Validation Accuracy")  # TODO: Plot validation accuracy for each epoch
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Set the model to evaluation mode
___

# Gather predictions & labels
all_preds = []
all_labels = []

# TODO: Disable gradient computation for evaluation
with ___:
    for inputs, labels in val_loader:
        # TODO: Move inputs and labels to device
        inputs = ___
        labels = ___
        
        # TODO: Get model predictions
        outputs = ___
        
        # TODO: Get predicted classes (use torch.max to find the class with the highest probability)
        _, preds = ___
        
        # TODO: Store predictions and labels (convert to CPU numpy arrays)
        all_preds.extend(___)
        all_labels.extend(___)

# TODO: Convert lists to numpy arrays
all_preds = ___
all_labels = ___

# TODO: Compute overall accuracy
overall_acc = ___
print(f"Overall Validation Accuracy: {overall_acc:.4f}")

# TODO: Create confusion matrix
cm = ___

# TODO: Create confusion matrix display
disp = ___

# TODO: Create and show the plot
fig, ax = plt.subplots(figsize=(12, 12))
# TODO: Plot the confusion matrix
___
plt.title("Confusion Matrix for Scene Classification")
plt.show()

In [ ]:
# QUESTION:
# Why apply transfer learning? Does fine-tuning improve results substantially? Why?